In [0]:

from pyspark.sql import functions as F

# ================================================
# STRAVL - MACHINE LEARNING
# ETAPA 1: PREPARAÇÃO DOS DADOS
# ================================================

# 1. Carregar Gold auditada
gold = spark.table(
    "workspace.gold.amostra_wide"
)

# 2. Selecionar avaliações binárias
# Sim = 1 | Não = 0
ml_base = (
    gold
    .filter(F.col("target").isin(0, 1))
    .select(
        "usuario_id",
        "destino_id",
        "destino",
        "target",
        "form_a_codigos",
        "form_b_codigos",
        "form_c_codigos",
        "form_f_codigos",
        "form_g_codigos",
        "form_h_codigos",
        "form_i_codigos",
        "form_j_codigos",
        "form_r_codigos",
        "form_rr_codigos",
        "orcamento_todas_faixas",
        "popularidade_todas_opcoes",
        "regiao_inconsistente"
    )
)

print("Total de avaliações para ML:", ml_base.count())
print("Total de variáveis:", len(ml_base.columns))

# 3. Separar viajantes, não avaliações
usuarios = (
    ml_base
    .select("usuario_id")
    .distinct()
)

usuarios_treino, usuarios_teste = (
    usuarios.randomSplit([0.8, 0.2], seed=42)
)

# 4. Criar bases de treino e teste
treino = ml_base.join(
    usuarios_treino,
    on="usuario_id",
    how="inner"
)

teste = ml_base.join(
    usuarios_teste,
    on="usuario_id",
    how="inner"
)

print("Avaliações de treino:", treino.count())
print("Avaliações de teste:", teste.count())

# 5. Validar que nenhum viajante
# aparece simultaneamente nos dois conjuntos
usuarios_repetidos = (
    usuarios_treino
    .join(
        usuarios_teste,
        on="usuario_id",
        how="inner"
    )
    .count()
)

print(
    "Viajantes presentes nos dois conjuntos:",
    usuarios_repetidos
)

assert usuarios_repetidos == 0

# 6. Distribuição do alvo
display(
    ml_base
    .groupBy("target")
    .count()
    .orderBy("target")
)
